# Build a Binary Classification Model


**Target:** Is the delivery `"On time"` (1) or `"Delayed"` (0)?

Before using complex algorithms, we need a baseline model to compare against. A simple baseline is the **majority-class classifier**, which always predicts the most frequent outcome.

_Why?_ If 90% of deliveries are on time, a model with 90% accuracy might actually be doing nothing more than guessing "on time" every time. We checked this baseline in the previous section and found that the majority class is "Delayed" at around 58%. So the baseline model would predict "Delayed" for every delivery, giving an accuracy of about 58%.


## Import Libraries

:::{important} Required packages

This notebook uses pandas, scikit-learn, xgboost, and plotly. All of them are listed in the `requirements.txt` file at the root of the repository. Install everything the book uses with:

```bash
python -m pip install -r requirements.txt
```

:::

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    accuracy_score,
)

import xgboost
from xgboost import XGBClassifier

import plotly.express as px
import plotly.io as pio

pio.templates.default = "simple_white"

Check XGBoost version.


In [2]:
print(f"xgboost version: {xgboost.__version__}")

xgboost version: 3.4.1


## Load Data


In [3]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/bdi593/datasets/main/amazon-on-time-delivery-data/on-time-delivery-data.csv"
)
df

,zipcode,total_items,precipitation_rate,water_runoff,snow_depth,temperature,temperature_at_1500m,min_temperature,max_temperature,pressure,wind_gust_speed,total_cloud_cover,dew_point_temperature,relative_humidity,wind_speed,classification_ontime
0,2719,518.0,0.0,1.80,0.0,2.24,-5.41,-3.32,4.36,102949.50,9.09,80.5,-2.44,71.58,4.892280,Delayed
1,92545,969.0,0.0,0.40,0.0,12.17,6.87,4.76,19.41,94921.35,3.16,0.0,-6.60,28.81,2.376716,Delayed
2,27357,216.0,0.0,2.05,0.0,5.30,0.92,0.41,11.21,98834.30,8.92,92.5,0.63,73.53,3.213254,On time
3,81211,402.0,0.0,0.13,0.0,-0.28,10.12,-3.24,5.16,70768.17,6.00,16.0,-5.72,67.84,3.751669,Delayed
4,67220,246.0,0.0,0.04,0.0,11.71,7.29,5.90,17.24,96777.72,15.99,5.0,5.49,66.28,9.623910,On time
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44995,11030,1228.0,0.0,0.00,0.0,5.64,2.97,3.52,6.85,101938.50,4.95,99.7,4.41,91.97,2.923090,Delayed
44996,63119,1627.0,0.0,6.10,0.0,8.46,2.25,3.33,14.21,100249.50,4.87,0.0,-1.65,51.47,2.134497,Delayed
44997,21032,496.0,0.0,2.14,0.0,3.53,-6.77,0.81,7.60,101679.00,7.78,5.7,-4.97,54.33,4.524221,Delayed
44998,95315,189.0,0.0,0.44,0.0,5.51,-2.32,-0.53,10.88,102243.50,2.73,15.7,-6.38,42.74,2.177973,On time


## Preprocess Data for Modeling


### Select Features and Target

We use every column except the target, with two changes found during EDA:

- **`precipitation_rate` is dropped.** It is 0 in every row, so it carries no information.
- **`zipcode` is replaced by `zip_region`.** A ZIP code is a label, not a quantity: ZIP code 90000 is not "more" than ZIP code 10000. The first digit of a US ZIP code identifies a broad group of states, so we keep that digit as a categorical feature. The column stores ZIP codes as integers, so a code such as 02719 appears as 2719, and integer division by 10,000 returns its first digit (0).

In [4]:
X = df.drop(columns=["classification_ontime", "precipitation_rate", "zipcode"])
X["zip_region"] = (df["zipcode"] // 10000).astype("category")

y = df["classification_ontime"]

X.dtypes

total_items               float64
water_runoff              float64
snow_depth                float64
temperature               float64
temperature_at_1500m      float64
min_temperature           float64
max_temperature           float64
pressure                  float64
wind_gust_speed           float64
total_cloud_cover         float64
dew_point_temperature     float64
relative_humidity         float64
wind_speed                float64
zip_region               category
dtype: object

### Encode Target Variable to Integer


In [5]:
le = LabelEncoder()
y = le.fit_transform(y)

print(dict(zip(le.classes_, le.transform(le.classes_))))

{'Delayed': np.int64(0), 'On time': np.int64(1)}


### Split Data into Train and Test Sets

The `random_state` parameter ensures that the split is reproducible, meaning you will get the same train and test sets every time you run the code. This is important for consistency when evaluating model performance. `42` is a commonly used value for `random_state`, but you can choose any integer.


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#### The Inputs

- `X`: Your feature matrix (independent variables).
  - Examples: `total_items`, `zip_region`, weather measurements
- `y`: Your target variable (dependent variable).
  - `"classification_ontime"`

#### The Arguments

- `test_size=0.2`: 20% of the data goes into the test set, and the remaining 80% goes into the training set.
  - Since we have 45,000 rows, this means:
    - 36,000 rows for training
    - 9,000 rows for testing
- `random_state=42`: Controls randomness.
  - Ensures the split is reproducible.
  - If someone else runs this code with `random_state=42`, they get the same split.
  - The number 42 is arbitrary (it's just a seed), although it's a popular choice in the data science community.
- `stratify=y`: Ensures that the class distribution in the train and test sets matches the original dataset.
  - Example: If your original dataset has:
    - 70% class 0
    - 30% class 1
  - Then both training and test sets will also have:
    - 70% class 0
    - 30% class 1
  - Without stratification, you might get an imbalanced split.
  - ⚠️ Only use `stratify=y` for classification tasks (not regression).

#### Outputs

The function returns 4 objects:

| Variable  | Meaning               |
| --------- | --------------------- |
| `X_train` | Features for training |
| `X_test`  | Features for testing  |
| `y_train` | Labels for training   |
| `y_test`  | Labels for testing    |


## Build Classifier


In [7]:
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    enable_categorical=True,
    random_state=42,
)

xgb_model.fit(X_train, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


### Model Arguments (Parameters)

The arguments you pass when creating the XGBoost model are called hyperparameters. They control how the model learns from the data. Choosing the right hyperparameters can significantly affect your model's performance.

| Parameter          | Type   | What It Controls                                          | Effect on Model                                                                    | Typical Range / Notes                         |
| ------------------ | ------ | --------------------------------------------------------- | ---------------------------------------------------------------------------------- | --------------------------------------------- |
| `n_estimators`     | int    | Number of boosting rounds (trees)                         | More trees → more complex model; can reduce bias but increase overfitting          | 100-1000 common                               |
| `learning_rate`    | float  | Step size shrinkage after each tree                       | Smaller values → slower learning, usually better generalization (needs more trees) | 0.01-0.3                                      |
| `max_depth`        | int    | Maximum depth of each tree                                | Deeper trees → capture more complex patterns but risk overfitting                  | 3-10 typical                                  |
| `subsample`        | float  | Fraction of training samples used per tree                | <1.0 adds randomness → reduces overfitting                                         | 0.5-1.0                                       |
| `colsample_bytree` | float  | Fraction of features used per tree                        | Adds feature randomness → reduces overfitting                                      | 0.5-1.0                                       |
| `objective`        | string | Loss function being optimized                             | Determines type of task (binary, multi-class, regression)                          | `"binary:logistic"` for binary classification |
| `eval_metric`      | string | Metric used to evaluate model performance during training | Does not affect training directly, just monitoring                                 | `"logloss"`, `"auc"`, `"error"` common        |
| `enable_categorical` | bool | Native support for pandas `category` columns            | Lets the trees split `zip_region` into groups of regions                           | Needed for `category` columns                 |
| `random_state`     | int    | Seed for reproducibility                                  | Ensures same random sampling each run                                              | Any fixed integer                             |

:::{note} These hyperparameters are optional

XGBoost has default values for all hyperparameters, so you can create a model without specifying any. However, tuning these hyperparameters can lead to better performance.

If you're new to XGBoost, it's a good idea to start with the default parameters and then experiment with tuning them later on.

:::


## Check Model Performance


After training the XGBoost classifier on the delivery dataset, we evaluate performance using several standard classification metrics.

Because this is a binary classification problem, no single metric tells the full story. Instead, we interpret accuracy, precision, recall, F1-score, and ROC-AUC together.


### Generate Predictions

After training the XGBoost model (`xgb_model`) on the training data, we use the test data (`X_test`) to generate predictions.

The `.predict()` method of the XGBoost model returns the predicted class labels (0 or 1) for the test set. In binary classification, it will output `0` for "Delayed" and `1` for "On time".

$$
\text{If } P(\text{class}=1) \ge 0.5 \rightarrow \text{predict } 1
$$

$$
\text{If } P(\text{class}=1) < 0.5 \rightarrow \text{predict } 0
$$


In [8]:
y_pred = xgb_model.predict(X_test)
y_pred

array([1, 1, 0, ..., 0, 0, 1], shape=(9000,))

:::{important} The threshold can vary

The default threshold for converting predicted probabilities to class labels using `.predict()` is 0.5. If you wish to use a different threshold (e.g., 0.7), you would need to use the `.predict_proba()` method to get the predicted probabilities and then apply your custom threshold to determine the class labels.

This can be useful if you want to be more conservative in predicting the positive class (e.g., only predict "On time" if the model is very confident), or if the classes are imbalanced. For example, fraud detection datasets typically have a very low percentage of positive cases (fraudulent transactions), so using a higher threshold can help reduce false positives.

:::


### Generate Predicted Probabilities

Instead of returning class labels, `predict_proba()` returns the predicted probabilities of the output being the positive class (class 1, "On time"). For example, if `predict_proba()` returns `0.8` for a delivery, it means the model estimates an 80% chance that the delivery will be "On time" and a 20% chance it will be "Delayed".


In [9]:
y_proba = xgb_model.predict_proba(X_test)[:, 1]
y_proba

array([0.9546095 , 0.8971429 , 0.10059921, ..., 0.06396656, 0.49449247,
       0.5563728 ], shape=(9000,), dtype=float32)

:::{tip} Why probabilities?

Probabilities give us more information than just class labels. They allow us to understand the model's confidence in its predictions. For instance, a prediction of `0.51` for "On time" is much less certain than a prediction of `0.95`. This can be useful for making decisions based on the level of risk or uncertainty.

:::


### Confusion Matrix

In binary classification, a confusion matrix is a $2 \times 2$ table used to evaluate the performance of a predictive model. It goes beyond simple accuracy by showing exactly where the model is succeeding and where it is miscalculating.

The matrix compares the actual target values with the predicted values generated by the model.

$$
\begin{bmatrix}
TN & FP \\
FN & TP
\end{bmatrix}
$$

| Term   | Full Name      | What It Means              | In Our Dataset                            |
| ------ | -------------- | -------------------------- | ----------------------------------------- |
| **TP** | True Positive  | Predicted 1 and actually 1 | Predicted on-time, and it arrived on time |
| **TN** | True Negative  | Predicted 0 and actually 0 | Predicted delayed, and it was delayed     |
| **FP** | False Positive | Predicted 1 but actually 0 | Predicted on-time, but it was delayed     |
| **FN** | False Negative | Predicted 0 but actually 1 | Predicted delayed, but it arrived on time |


In [10]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Confusion Matrix:
[[4382  846]
 [ 698 3074]]


#### Interpreting the Confusion Matrix

$$
\begin{bmatrix}
4382 & 846 \\
698 & 3074
\end{bmatrix}
$$

- **4382 (True Negatives, TN):** The model correctly predicted "Delayed" for 4,382 deliveries.
- **846 (False Positives, FP):** The model incorrectly predicted "On time" for 846 deliveries that were actually "Delayed".
- **698 (False Negatives, FN):** The model incorrectly predicted "Delayed" for 698 deliveries that were actually "On time".
- **3074 (True Positives, TP):** The model correctly predicted "On time" for 3,074 deliveries.

### Accuracy

Accuracy is the proportion of correct predictions out of all predictions.

$$
\text{Accuracy} = \frac{\text{Number of Correct Predictions}}{\text{Total Predictions}}
$$

In classification, we can use the confusion matrix to break down the predictions into four categories.

$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$


In [11]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.8284444444444444


If you randomly pick one delivery, there's about an 83% chance that the model will correctly predict whether it was "On time" or "Delayed".

> "Out of all deliveries, how often was the model right?"

:::{important} Accuracy can be misleading

Accuracy can be misleading, especially when dealing with imbalanced datasets. For example, if 90% of deliveries are "Delayed", a model that always predicts "Delayed" will have 90% accuracy, but it would be useless for identifying "On time" deliveries. In such cases, other metrics like precision, recall, and F1-score can provide a more nuanced view of model performance.

:::


### ROC-AUC

ROC-AUC measures how well your model separates the two classes across all possible thresholds.

It stands for:

- ROC = Receiver Operating Characteristic
- AUC = Area Under the Curve

The ROC curve plots True Positive Rate (TPR) and False Positive Rate (FPR) for every possible threshold.

#### True Positive Rate (Recall)

$$
\text{True Positive Rate (TPR)} = \frac{\text{TP}}{\text{TP} + \text{FN}}
$$

This measures the proportion of actual positive cases (on-time deliveries) correctly identified.

#### False Positive Rate

$$
\text{False Positive Rate (FPR)} = \frac{\text{FP}}{\text{FP} + \text{TN}}
$$

This measures the proportion of actual negative cases (delayed deliveries) that were incorrectly predicted as positive.

AUC is the area under the ROC curve, which ranges from 0 to 1. A model with an AUC of 0.5 has no discriminative ability (equivalent to random guessing), while a model with an AUC of 1.0 perfectly separates the classes.

| AUC     | Interpretation  |
| ------- | --------------- |
| 0.5     | Random guessing |
| 0.7-0.8 | Fair            |
| 0.8-0.9 | Good            |
| 0.9-1.0 | Excellent       |
| 1.0     | Perfect         |


In [12]:
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

ROC-AUC: 0.9075823265052116


$$
\text{ROC-AUC} \approx 0.908
$$

This means that the model has excellent ability to distinguish between "On time" and "Delayed" deliveries across all thresholds.

If you randomly select one on-time delivery and one delayed delivery, the model assigns a higher predicted probability to the on-time delivery about 90.8% of the time.


:::{seealso} Accuracy vs ROC-AUC

Accuracy answers:

> "How often are we correct at threshold 0.5?"

ROC-AUC answers:

> "How well does the model separate the two classes overall?"

:::


### Other Metrics

The `classification_report()` function from scikit-learn provides additional metrics like precision, recall, and F1-score for each class. These metrics can give you more insight into the performance of your model.


In [13]:
print("Classification Report:")
print(classification_report(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.84      0.85      5228
           1       0.78      0.81      0.80      3772

    accuracy                           0.83      9000
   macro avg       0.82      0.83      0.82      9000
weighted avg       0.83      0.83      0.83      9000



#### Precision

$$
\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}
$$

Precision answers:

> When the model predicts this class, how often is it correct?

- When the model predicts "delayed", it is correct 86% of the time.
- When the model predicts "on-time", it is correct 78% of the time.
  - This is lower - meaning some delayed deliveries are being predicted as on-time (false positives).


#### Recall

$$
\text{recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}
$$

Recall answers:

> Out of all actual cases of this class, how many did we correctly identify?

- The model correctly catches 84% of delayed deliveries (0).
- The model correctly identifies 81% of on-time deliveries (1).


#### F1-Score

$$
\text{F1-Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}
$$

F1 is the harmonic mean of precision and recall. It balances false positives and false negatives. A high F1-score means the model has both high precision and high recall. In our case, the F1-scores for both classes are around 0.80-0.85, indicating good performance in identifying both on-time and delayed deliveries.

- For class 0 (delayed), the F1-score is around 0.85, which indicates a strong overall performance for delayed deliveries.
- For class 1 (on-time), the F1-score is around 0.80, which is slightly lower than for delayed deliveries. This is weaker than the F1-score for delayed deliveries, but still solid.


#### Support

Support = number of actual observations in each class.

- Class 0 (Delayed): 5228
- Class 1 (On-time): 3772

This tells you the dataset is moderately imbalanced (58% delayed, 42% on-time).


#### 🎯 Business Interpretation

Your model:

- Is stronger at predicting delays than on-time deliveries
- Has slightly lower precision for predicting on-time (0.78)
- Has similar recall for both classes (0.84 for delayed, 0.81 for on-time)
- Shows strong overall discrimination (AUC≈0.91)


## Check Feature Importance


In [14]:
importances = xgb_model.feature_importances_
features = X.columns

df_feat_imp = pd.Series(importances, index=features).sort_values().reset_index()

df_feat_imp.columns = ["feature", "importance"]
df_feat_imp["importance_pct"] = df_feat_imp["importance"] * 100

df_feat_imp

,feature,importance,importance_pct
0,relative_humidity,0.018330,1.832962
1,total_cloud_cover,0.018668,1.866849
2,water_runoff,0.019247,1.924684
3,wind_gust_speed,0.019377,1.937704
4,wind_speed,0.021247,2.124679
5,temperature_at_1500m,0.023256,2.325629
6,pressure,0.025737,2.573663
7,snow_depth,0.033315,3.331519
8,min_temperature,0.034005,3.400506
9,zip_region,0.034437,3.443718


In [15]:
fig = px.bar(
    df_feat_imp,
    x="importance_pct",
    y="feature",
    orientation="h",
    title="XGBoost Feature Importance (gain)",
    labels={"importance_pct": "Importance (%)", "feature": "Feature"},
    height=500,
    template="simple_white",
)
fig.show()

### Why Does `total_items` Have Such High Feature Importance?

One striking result is that `total_items` accounts for most of the model's total feature importance. Before looking for a business story (more items mean more picking and packing time), look at how the label is defined.

According to the source of this dataset, a delivery is labeled **Delayed** when, on a specific day, **at least 10 items from one truck** arrived late, and `total_items` is the number of items on that truck. A truck with 2,000 items has many more chances to reach 10 late items than a truck with 150 items, even if every item has the same small chance of being late. So `total_items` is tied to the label by the way the label is built.

We can check this directly. Split the deliveries into 10 equal-sized groups (deciles) by `total_items`, and compute the share of delayed deliveries in each group.

In [16]:
df_deciles = (
    df.assign(
        items_decile=pd.qcut(df["total_items"], 10, labels=False) + 1,
        delayed=(df["classification_ontime"] == "Delayed").astype(int),
    )
    .groupby("items_decile")
    .agg(
        min_items=("total_items", "min"),
        max_items=("total_items", "max"),
        delay_rate=("delayed", "mean"),
    )
)

df_deciles.round(3)

,min_items,max_items,delay_rate
items_decile,,,
1,102.0,135.0,0.121
2,136.0,184.0,0.159
3,185.0,253.0,0.229
4,254.0,353.0,0.343
5,354.0,491.0,0.511
6,492.0,666.0,0.689
7,667.0,888.0,0.834
8,889.0,1198.0,0.940
9,1199.0,1726.0,0.986


In [17]:
fig = px.bar(
    df_deciles.reset_index(),
    x="items_decile",
    y="delay_rate",
    title="Share of delayed deliveries by total_items decile",
    labels={"items_decile": "total_items decile (1 = fewest items)", "delay_rate": "Share delayed"},
    template="simple_white",
)
fig.update_xaxes(dtick=1)
fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.show()

The share of delayed deliveries climbs steadily from the lowest decile to almost 100% in the highest. That suggests a one-line rule: predict "Delayed" whenever `total_items` is above some cut-off.

### A One-Feature Baseline

A fair baseline must be built the same way as the model: choose the cut-off on the training set only, then score it once on the test set.

In [18]:
# Try every distinct total_items value in the training set as a cut-off.
# Rule: predict "On time" (1) when total_items <= cut-off, otherwise "Delayed" (0).
train_items = X_train["total_items"].to_numpy()
thresholds = np.unique(train_items)
train_accuracy = [np.mean((train_items <= t) == y_train) for t in thresholds]
best_threshold = thresholds[int(np.argmax(train_accuracy))]
print(f"Best cut-off on the training set: total_items <= {best_threshold:.0f}")

y_pred_rule = (X_test["total_items"] <= best_threshold).astype(int)

comparison = pd.DataFrame(
    {
        "accuracy": [
            accuracy_score(y_test, np.zeros_like(y_test)),
            accuracy_score(y_test, y_pred_rule),
            accuracy_score(y_test, y_pred),
        ],
        # Ranking deliveries by -total_items gives the ROC AUC of the feature alone
        "roc_auc": [
            np.nan,
            roc_auc_score(y_test, -X_test["total_items"]),
            roc_auc_score(y_test, y_proba),
        ],
    },
    index=[
        "Majority class (always Delayed)",
        f"One rule: On time if total_items <= {best_threshold:.0f}",
        f"XGBoost ({X.shape[1]} features)",
    ],
)

comparison.round(3)

Best cut-off on the training set: total_items <= 422


,accuracy,roc_auc
Majority class (always Delayed),0.581,NaN
One rule: On time if total_items <= 422,0.811,0.887
XGBoost (14 features),0.828,0.908


The one-feature rule gets within about two points of XGBoost's accuracy, and it already clears the 80% goal set in the case overview. Ranking deliveries by `total_items` alone gives a ROC AUC only slightly below the model's. Almost all of the model's skill comes from one column that is part of the label's definition. The weather and location features add a little on top.

:::{important} Predictive importance is not causal influence

It is tempting to explain the importance of `total_items` with logistics stories: bigger loads need more picking and packing time, more stops, or heavier packaging. Some of that may be true, but the label definition alone is enough to produce this pattern. Before you trust a story about a dominant feature, check whether the feature is built into the target.

This matters for the business question. If the goal is to learn how weather affects deliveries, a target that grows with the number of items hides the weather signal. A better target would be the *share* of a truck's items that arrived late, which does not rise mechanically with load size. With the label we have, any model of this data is mostly a model of truck size.

:::

## Summary

- We dropped the constant `precipitation_rate`, replaced `zipcode` with a categorical `zip_region`, and trained an XGBoost classifier on an 80/20 stratified split.
- On the test set the model reaches about 83% accuracy and a ROC AUC of about 0.91, well above the 58% majority-class baseline.
- `total_items` dominates feature importance because the label counts late items per truck. A one-feature rule on `total_items` reaches about 81% accuracy, close to the full model.
- The next notebook tries common techniques to improve the model and measures each one on the same test set.